# Labwork 4 — Second-order methods and the statistics of the loss

**Week 2 · Day 4 · ≈ 170 min at the keyboard**

Read Lecture 4 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** Cholesky, Newton, and the regularizer — after which ridge regression costs you nothing

**Files you will open:**

- `linalg/cholesky.py`
- `problems/glm.py`, `quadratic.py`, `rosenbrock.py` (the `hessian` methods)
- `numerics/gradcheck.py` (`check_hessian`)
- `optimizers/directions.py`
- `regularizers.py`, `objective_ops.py`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 1 — Cholesky  *(≈ 55 min)*

Implement `cholesky`, `solve_lower`, `solve_upper_from_lower` and `CholeskySolver`.

Vectorize the inner loop over `i` — three nested Python loops will be unusably slow on the
day-6 benchmark.

**Raise `NotPositiveDefiniteError` as soon as a pivot is non-positive.** Do not clamp it,
do not add a small epsilon, do not fall back to `np.linalg.solve`. The failure carries
information and the caller needs to see it.

Three checks: the hand example `[[4,2,2],[2,5,3],[2,3,6]] → L = [[2,0,0],[1,2,0],[1,1,2]]`;
agreement with `np.linalg.solve` on random SPD matrices; and the refusal on
`[[1,2],[2,1]]`, whose second pivot would need `√(−3)`.

Then try a Hilbert matrix. The residual `‖Ax − b‖` will be tiny while the error
`‖x − x_true‖` is enormous — a small residual does **not** mean an accurate answer.

**Open:** `src/optlab/linalg/cholesky.py`

In [ ]:
edit("linalg/cholesky.py")
check("tests/unit/test_day4_cholesky.py", "tests/contracts/test_linear_solver_contract.py")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`L[j+1:, j] = (A[j+1:, j] - L[j+1:, :j] @ L[j, :j]) / L[j, j]` does the whole column at once. Forward substitution: `y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]`.

</details>

---

## Exercise 2 — Hessians and Newton  *(≈ 55 min)*

Implement `check_hessian` (`gradcheck.py`), the three `hessian` methods, then
`NewtonDirection` and the `newton()` assembly.

`check_hessian` should **reuse** `numerical_jacobian` from day 1 — a Hessian *is* the
Jacobian of the gradient. Do not write a second finite-difference routine.

`GLMLoss.hessian` is `XᵀDX / n` with `D = diag(φ''(Xw, y))`. Get that as one vectorized
expression, not a loop over samples.

`NewtonDirection` receives its solver by constructor and calls `solve(H, -g)`. Let
`NotPositiveDefiniteError` propagate — Exercise 3 deals with it.

Four things to verify:

1. **Linear regression converges in exactly one iteration**, matching `np.linalg.solve` on
   the normal equations.
2. Logistic regression reaches the same minimizer as your day-2 gradient descent.
3. The error ratio `e_{k+1}/e_k²` stays bounded — that is quadratic convergence.
4. **You did not edit `descent.py`.** `newton()` is four existing objects wired together.

**Open:** `src/optlab/numerics/gradcheck.py`, `problems/*.py`, `optimizers/directions.py`, `optimizers/descent.py`

In [ ]:
check("-m", "day4")

import subprocess
d = subprocess.run(["git", "diff", "--stat", "src/optlab/optimizers/descent.py"],
                   cwd=OPTLAB, capture_output=True, text=True).stdout.strip()
print("\nchanges to the day-2 loop:", d if d else "none -- Newton needed no new loop")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

For the GLM Hessian: `X.T @ (X * d2[:, None]) / n`. Broadcasting scales each row of `X` by its weight, which is exactly `XᵀDX` without ever forming `D`.

</details>

---

## Exercise 3 — Damping, and a failure worth seeing  *(≈ 30 min)*

Newton's method does not look for a minimum. It looks for a point where `∇f = 0`, and a
maximum satisfies that just as well as a minimum does.

Watch it happen on the double well `f(x) = (x² − 1)²`, whose minima are at `x = ±1` and
whose maximum is at `x = 0`. Start at `x₀ = 0.3`, where `f''(0.3) = −2.92 < 0`, and write
the scalar iteration `x ← x − f'(x)/f''(x)` out by hand in the notebook — three lines of
numpy, no optlab. You will land on `x = 0` in about three steps and stay there: Newton
has *converged*, to the worst point in the picture.

Now the same problem through your own code, and the first surprise:

```python
DescentOptimizer(NewtonDirection(CholeskySolver()), FixedStep(1.0), stop).minimize(...)
```

This does **not** reproduce the walk to the maximum. It raises `NotPositiveDefiniteError`
on the very first iteration, because `CholeskySolver` refuses the indefinite Hessian
before any step is taken. That refusal is the design decision you implemented in
Exercise 1, working exactly as intended — the method you wrote is *safer* than the
textbook iteration, and it told you so by raising.

Then the second surprise. Try to repair it with a line search:

```python
DescentOptimizer(NewtonDirection(CholeskySolver()), Armijo(), stop).minimize(...)
```

**This also raises, and the error is identical.** A line search chooses how *far* to go
along a direction; it cannot help when no direction was produced at all. Armijo never gets
to run. Be able to state that distinction — step length against step direction — because
it is the reason the next repair has to live in the `DirectionRule`, not in the
`LineSearch`.

So fix the direction. Fill in `ModifiedNewton` in the same file: catch
`NotPositiveDefiniteError`, solve `(H + τI)p = −g` instead, and double `τ` until Cholesky
succeeds. Read its docstring first — it tells you why τ restarts from `tau0` on every call
rather than persisting, and why exceeding `tau_max` should raise rather than quietly fall
back to `−g`.

Verify, with a `History` observer:

- `ModifiedNewton` + `Armijo` from `x₀ = 0.3` reaches `x = +1` and stops there;
- the recorded values are *strictly* decreasing, every step;
- it takes only a handful of iterations — the damping is active for the first step or two
  and then gets out of the way, because once you are near `x = 1` the Hessian is positive
  definite again and `τ` never has to leave `tau0`.

**When is `Armijo` the right repair, then?** When the Hessian *is* positive definite but
the full step `α = 1` overshoots — far out on Rosenbrock, for instance. Try it there and
see the difference. Two different failures, two different fixes: an indefinite Hessian is
a direction problem, an overshooting step is a length problem. Reaching for the wrong one
does nothing at all, which you have now seen.

`H + τI` is worth remembering overnight. Tomorrow it returns as Levenberg–Marquardt, where
τ is called λ and is chosen by a trust-region rule instead of by doubling.

**Open:** `src/optlab/optimizers/directions.py`


In [ ]:
check("-m", "day4")

---

## Exercise 4 — Ridge, for free  *(≈ 30 min)*

Implement `NoRegularizer` and `L2` (`value = ½λ‖w‖²`, `gradient = λw`,
`prox = w/(1+λt)`), then `RegularizedObjective` — an adapter presenting `f + r` as a single
`Objective`.

Now the payoff. Without writing a single new optimizer:

```python
ridge = RegularizedObjective(GLMLoss(X, y, SquaredError()), L2(lam))
newton().minimize(ridge, w0)          # ridge regression
gradient_descent().minimize(ridge, w0)  # also ridge regression
```

Every optimizer you have written gained a regularized version the moment this adapter
existed. Check the Newton solution against the closed form `(XᵀX/n + λI)⁻¹Xᵀy/n`.

Then look at the eigenvalues of `H` and `H + λI`: every one lifted by exactly `λ`. That is
why ridge can never break Cholesky, and it is the last piece of day 5.

**Open:** `src/optlab/regularizers.py`, `src/optlab/objective_ops.py`

In [ ]:
check("tests/contracts/test_regularizer_contract.py", "-m", "day4")

---

## Checkpoint

Everything from day 1 to day 4 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2 or day3 or day4")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Why not invert `H`? Give both the flop count and the numerical reason.
2. What does "Cholesky failed" mean **geometrically**, and what does it mean
   **statistically**? They are the same sentence in two languages.
3. Your `CholeskySolver` is about to be reused twice tomorrow without modification. Which
   design decision made that possible?